# Gaussian Process reproduction

Core model/preprocessing logic is derived from `linphotonicslab/ML_Pipeline` (MIT License).  
This portfolio notebook records my independent rerun of the published workflow; it does **not** claim the Gaussian Process architecture as an original implementation.

My work here is the environment adaptation, rerunning Optuna/model evaluation, and verification against the published workflow. See `../README.md` and `../REPRODUCTION_RESULTS.md` for contribution boundaries.


In [ ]:
%reset -f
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RationalQuadratic, Matern, WhiteKernel, RBF, Sum
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import numpy as np
import pandas as pd
import optuna
import warnings

OUTPUT_TEST = True


In [ ]:
# Load training and validation datasets.
X_train = pd.read_csv("../data/cleaned/training.csv")
y_train = pd.read_csv("../data/cleaned/training_labels.csv")
X_val = pd.read_csv("../data/cleaned/validation.csv")
y_val = pd.read_csv("../data/cleaned/validation_labels.csv")

# Replace brackets in column names for sklearn compatibility.
for col in list(X_train.columns):
    if "[" in col or "]" in col:
        new_col = col.replace("[", "(").replace("]", ")")
        X_train = X_train.rename(columns={col: new_col})
        X_val = X_val.rename(columns={col: new_col})

# Verification split for Optuna tuning.
X_train, X_verif, y_train, y_verif = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)
for obj in (X_train, y_train, X_verif, y_verif, X_val, y_val):
    obj.reset_index(drop=True, inplace=True)


In [ ]:
def define_kernel(trial):
    kernels = []
    n_kernels = trial.suggest_int("n_kernels", 1, 3)

    for i in range(n_kernels):
        kernel_type = trial.suggest_categorical(
            f"kernel_type_{i}", ["Matern", "RationalQuadratic"]
        )
        if kernel_type == "RationalQuadratic":
            length_scale = trial.suggest_float(
                f"RationalQuadratic_{i}_length_scale", 1e-1, 1e5
            )
            alpha = trial.suggest_float(
                f"RationalQuadratic_{i}_alpha", 1e-1, 1e5
            )
            kernel = RationalQuadratic(
                length_scale=length_scale,
                alpha=alpha,
                length_scale_bounds=(1e-8, 1e8),
            )
        else:
            length_scale = trial.suggest_float(
                f"Matern_{i}_length_scale", 1e-1, 1e5
            )
            nu = trial.suggest_float(f"Matern_{i}_nu", 0.5, 5)
            kernel = Matern(
                length_scale=length_scale,
                nu=nu,
                length_scale_bounds=(1e-8, 1e8),
            )
        kernels.append(kernel)

    # Kept from the reproduced search space even though WhiteKernel is not added below.
    trial.suggest_float(f"WhiteKernel_{n_kernels}_noise_level", 1e-5, 1e5)

    combined = kernels[0]
    for kernel in kernels[1:]:
        combined = Sum(combined, kernel)
    return combined


def objective(trial):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-3, 1e3, log=True),
        "n_restarts_optimizer": trial.suggest_int("n_restarts_optimizer", 0, 10),
        "kernel": define_kernel(trial),
    }

    model = GaussianProcessRegressor(**params)

    # The reproduced notebook tunes on a 500-row random training subset.
    X_sample = X_train.sample(n=min(500, len(X_train)))
    y_sample = y_train.loc[X_sample.index]
    model.fit(X_sample, y_sample)

    pred = model.predict(X_verif)
    mape = mean_absolute_percentage_error(y_verif, pred) * 100
    rmse = mean_squared_error(y_verif, pred, squared=False)
    return mape + rmse


In [ ]:
warnings.filterwarnings("ignore", category=RuntimeWarning)
study = optuna.create_study(pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

trial = study.best_trial
print("Best value:", trial.value)
print("Best parameters:")
for key, value in trial.params.items():
    print(f"  {key}: {value}")


In [ ]:
def reconstruct_kernel(params):
    kernels = []
    n_kernels = params["n_kernels"]
    for i in range(n_kernels):
        kind = params[f"kernel_type_{i}"]
        length_scale = params[f"{kind}_{i}_length_scale"]
        if kind == "Matern":
            kernel = Matern(
                length_scale=length_scale,
                nu=params[f"{kind}_{i}_nu"],
            )
        else:
            kernel = RationalQuadratic(
                length_scale=length_scale,
                alpha=params[f"{kind}_{i}_alpha"],
            )
        kernels.append(kernel)

    combined = kernels[0]
    for kernel in kernels[1:]:
        combined = Sum(combined, kernel)
    return combined


params = trial.params
kernel = reconstruct_kernel(params)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=params["alpha"],
    n_restarts_optimizer=params["n_restarts_optimizer"],
)

# The original reproduced workflow used the first 499 training rows for the final GP fit.
gp.fit(X_train.iloc[:499], y_train.iloc[:499])

val_pred = gp.predict(X_val)
print("Validation RMSE:", mean_squared_error(y_val, val_pred, squared=False))
print("Validation R²:", r2_score(y_val, val_pred))


In [ ]:
if not OUTPUT_TEST:
    raise ValueError("Set OUTPUT_TEST=True to run the held-out test set.")

X_test = pd.read_csv("../data/cleaned/test.csv")
y_test = pd.read_csv("../data/cleaned/test_labels.csv")

for col in list(X_test.columns):
    if "[" in col or "]" in col:
        X_test = X_test.rename(
            columns={col: col.replace("[", "(").replace("]", ")")}
        )

test_preds = gp.predict(X_test)
train_preds = gp.predict(X_train)

from pathlib import Path
pred_dir = Path("../data/predictions/GP")
pred_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(test_preds).to_csv(pred_dir / "test_pred_gp.csv", index=False, header=False)
pd.DataFrame(y_test).to_csv(pred_dir / "test_true_gp.csv", index=False, header=False)
pd.DataFrame(train_preds).to_csv(pred_dir / "train_pred_gp.csv", index=False, header=False)
pd.DataFrame(y_train).to_csv(pred_dir / "train_true_gp.csv", index=False, header=False)
pd.DataFrame(X_train).to_csv(pred_dir / "train_input_gp.csv", index=False, header=False)
pd.DataFrame(X_test).to_csv(pred_dir / "test_input_gp.csv", index=False, header=False)

test_true = np.asarray(y_test).ravel()
print("Test RMSE:", mean_squared_error(test_true, test_preds, squared=False))
print("Test R²:", r2_score(test_true, test_preds))
print("Test MAPE (%):", mean_absolute_percentage_error(test_true, test_preds) * 100)
